# Retention Copilot: run the agent and the evaluation on Kaggle

Settings (right panel): Accelerator = *GPU T4 x2*, *Internet = On*. Run the cells ONE AT A TIME (never 'Run All').

1-2 install and start the model. 3 gets the code. 4 smoke test (read the drafts). 5 mini-evaluation (8 customers, a few minutes). 6 optional: a bigger model. 7 the full evaluation. 8 package results.

If a cell errors, paste the error back.

In [ ]:
GITHUB_USER = "Randeep-Sidhu"
MODEL = "granite4:micro"

# Install Ollama (Internet must be On)
!apt-get install -y -q zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start the server in the background and download the model
import subprocess, time
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)
!ollama pull {MODEL}

In [ ]:
# Get the code (works whether or not the repo has a nested folder), install extras, run the offline tests
import os, pathlib
%cd /kaggle/working
!rm -rf retention-copilot
!git clone https://github.com/{GITHUB_USER}/Retention-copilot.git retention-copilot
root = next(p.parent.parent for p in pathlib.Path("retention-copilot").rglob("src/agent.py"))
os.chdir(root)
print("project root:", root)
!pip install -q langgraph langchain-ollama fastembed pytest
!python -m pytest -q 2>&1 | tail -5

In [ ]:
# Smoke test: 3 customers through the full agent, then 2 through the no-policy baseline. Read the drafts!
!python -m src.agent --customer auto --n 3 --config rag+verify --no-persist --model {MODEL}
!python -m src.agent --customer auto --n 2 --config none --no-persist --model {MODEL}

In [ ]:
# Mini-evaluation: 8 customers x 3 configurations (a few minutes). --fresh clears earlier eval runs.
!python -m src.evaluate --model {MODEL} --n 8 --configs none,rag,rag+verify --fresh
!head -40 reports/agent_eval_granite4-micro.md

In [ ]:
# OPTIONAL: try IBM's larger Granite model (downloads ~19 GB; needs both GPUs). Same mini-evaluation.
BIG = "granite4:small-h"
!ollama pull {BIG}
!python -m src.evaluate --model {BIG} --n 8 --configs none,rag,rag+verify
!head -40 reports/agent_eval_granite4-small-h.md

In [ ]:
# Full evaluation: 6 configurations x 48 held-out customers (resumable: re-run this cell if the session drops)
!python -m src.evaluate --model {MODEL} --n 48

In [ ]:
# Package the results, then download results.zip from the Output panel (/kaggle/working)
import zipfile, pathlib
root = pathlib.Path.cwd()
with zipfile.ZipFile("/kaggle/working/results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in [root / "db" / "eval_runs.db", *root.glob("reports/**/*")]:
        if p.is_file():
            z.write(p, p.relative_to(root))
print("results.zip written")